# Lesson 4: Persistence and Streaming

In [2]:
# 基本和Lesson2一致，不一样的地方代码注释讲解，其他参考Lesson2的代码注释
from dotenv import load_dotenv

_ = load_dotenv()

In [3]:
import os
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_tavily import TavilySearch

In [4]:
tool = TavilySearch(max_results=2)

In [5]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [7]:
class Agent:
    """
    一个基于LangGraph实现的ReAct风格Agent，支持状态持久化
    ReAct = Reasoning（思考） + Acting（行动）
    """
    def __init__(self, model, tools, checkpointer, system=""):
        """
        初始化Agent
        
        :param model: 语言模型（如ChatTongyi）
        :param tools: 工具列表（如[TavilySearch]）
        :param checkpointer: 状态检查点（用于持久化）
        :param system: 系统提示词（指导AI行为）
        """
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        # 编译图，启用状态持久化
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [53]:

# 提示词翻译：
# 你是一位聪明的研究助理。请使用搜索引擎查找信息。
# 您可以进行多次通话（可以同时进行，也可以按顺序进行）。
# 只有在确定自己想要什么的时候才去查找信息。
# 如果你在提出后续问题之前需要查找一些信息，这是允许的！
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatTongyi(
    model="qwen-turbo",  # 或其他通义千问模型
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"),  # 通义千问 API key
    temperature=0, 
    streaming=True
)

In [9]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# 内存模式 - 数据仅存在于内存中
# 允许跨线程使用
conn = sqlite3.connect(":memory:", check_same_thread=False)
checkpointer = SqliteSaver(conn)
abot = Agent(model, [tool], system=prompt, checkpointer=checkpointer)
# 下面的方式也可以，但是要搭配with使用，
# checkpointer = SqliteSaver.from_conn_string(":memory:")
# with SqliteSaver.from_conn_string(":memory:") as checkpointer:
#     # 在这里使用 checkpointer
#     abot = Agent(model, [tool], system=prompt, checkpointer=checkpointer)
#     for event in abot.graph.stream({"messages": messages}, thread):
#         for v in event.values():
#             print(v['messages'])
# 为什么需要状态持久化？
# 传统AI对话：
#   用户提问 → AI回答 → 对话结束（忘记之前内容）
# 有状态的AI对话：
#   用户提问 → AI回答 → 保存状态 → 下次提问能记住之前内容

# SqliteSaver工作原理
# 1. :memory: 表示在内存中创建SQLite数据库（重启后消失）
#    - 实际应用中应使用文件路径（如"checkpoints.db"）
# 2. 它会自动保存：
#    - 每个会话的完整对话历史
#    - 当前执行到哪一步（节点状态）
#    - 中间结果
# 3. 通过thread_id区分不同用户的会话

# 【安全提示】
# 1. 内存数据库(:memory:)仅用于教学演示
# 2. 生产环境应使用：
#    - 持久化文件（如"checkpoints.db"）
#    - 或专业数据库（PostgreSQL等）
# 3. 确保数据库文件不在Git仓库中（添加到.gitignore）

In [10]:
from langchain_core.runnables import RunnableConfig
# 设置会话ID（关键！）
# 写法1：简化版
# config = {"configurable": {"thread_id": "1"}}
# 写法2：带类型注解 官方推荐
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

# thread_id的作用
# 1. 区分不同用户的会话
#    - thread_id="1" → 用户A的会话
#    - thread_id="2" → 用户B的会话
# 2. LangGraph通过它：
#    - 加载之前的对话状态
#    - 继续之前的对话流程
#    - 避免混淆不同用户的对话
# 3. 实际应用中：
#    - 可以用用户ID作为thread_id
#    - 或会话UUID

# 【重要提示】
# 没有thread_id，每次对话都是全新的（无法记住之前内容）
# 有thread_id，Agent能记住之前的对话（真正的多轮对话）

In [11]:
messages = [HumanMessage(content="What is the weather in sf?")]

# 流式执行并打印结果
for event in abot.graph.stream({"messages": messages}, config):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"query": "What is the weather in sf?"}', 'name': 'tavily_search'}, 'id': 'call_afbccb1467274103b3c88c', 'index': 0, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-turbo', 'finish_reason': 'tool_calls', 'request_id': 'f8d61d48-888a-4830-ada9-e1a1556bd6e1', 'token_usage': {'input_tokens': 1874, 'output_tokens': 27, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 1901}}, id='lc_run--483b3468-4da5-4adf-afb8-907c8e35bf7a-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'What is the weather in sf?'}, 'id': 'call_afbccb1467274103b3c88c', 'type': 'tool_call'}])]
Calling: {'name': 'tavily_search', 'args': {'query': 'What is the weather in sf?'}, 'id': 'call_afbccb1467274103b3c88c', 'type': 'tool_call'}
Back to the model!
[ToolMessage(content='{\'query\': \'What is the weather in sf?\', \'follow_up_questions\': None, \'answer\': None, \'images\': [], \'results\': [{\'tit

In [12]:
messages = [HumanMessage(content="What about in la?")]

# 流式执行并打印结果
for event in abot.graph.stream({"messages": messages}, config):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"query": "What is the weather in la?"}', 'name': 'tavily_search'}, 'id': 'call_f85ca1843f9b473b9b07e6', 'index': 0, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-turbo', 'finish_reason': 'tool_calls', 'request_id': 'c3366aae-8fc8-421b-917f-62609b5c5ca4', 'token_usage': {'input_tokens': 2905, 'output_tokens': 27, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 2932}}, id='lc_run--8508ef98-c3ce-4382-a7a8-4191d3d8cbcf-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'What is the weather in la?'}, 'id': 'call_f85ca1843f9b473b9b07e6', 'type': 'tool_call'}])]}
Calling: {'name': 'tavily_search', 'args': {'query': 'What is the weather in la?'}, 'id': 'call_f85ca1843f9b473b9b07e6', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='{\'query\': \'What is the weather in la?\', \'follow_up_questions\': None, \'answer\': None, \'images\

In [13]:
messages = [HumanMessage(content="What one is warmer?")]

# 流式执行并打印结果
for event in abot.graph.stream({"messages": messages}, config):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='Based on the current weather data:\n\n- **San Francisco (SF)**: 15.6°C (60.1°F)\n- **Los Angeles (LA)**: 19.4°C (66.9°F)\n\n**Los Angeles (LA)** is warmer than San Francisco (SF).', additional_kwargs={}, response_metadata={'model_name': 'qwen-turbo', 'finish_reason': 'stop', 'request_id': '362506b5-57e1-4aff-a545-dab22d4ff69c', 'token_usage': {'input_tokens': 3839, 'output_tokens': 63, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 3902}}, id='lc_run--65c27819-abf1-4826-9311-404cdb93df17-0')]}


In [14]:
messages = [HumanMessage(content="What one is warmer?")]
config2: RunnableConfig = {"configurable": {"thread_id": "2"}}
# 流式执行并打印结果
for event in abot.graph.stream({"messages": messages}, config2):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='The question "What one is warmer?" is unclear. Could you please provide more context or clarify what you\'re asking? For example, are you comparing two objects, places, or something else?', additional_kwargs={}, response_metadata={'model_name': 'qwen-turbo', 'finish_reason': 'stop', 'request_id': '195f54ed-c5c3-4ac3-a997-9f625d19f200', 'token_usage': {'input_tokens': 1872, 'output_tokens': 39, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 1911}}, id='lc_run--4c9ccc3b-047b-4b62-bf7b-549c4459434e-0')]}


## Streaming tokens

## AsyncSqliteSaver 和 SqliteSaver 的区别
`SqliteSaver` 和 `AsyncSqliteSaver` 的核心区别在于它们处理数据库 I/O 的**方式**，即**同步（Synchronous）** 和**异步（Asynchronous）**，在 LangGraph 的语境中，它们是用于实现状态持久化（Checkpointer）的两种模式，必须与相应的执行方法匹配。

| **特性**       | **SqliteSaver**                                                         | **AsyncSqliteSaver**                                                                       |
| ------------ | ----------------------------------------------------------------------- | ------------------------------------------------------------------------------------------ |
| **I/O 模式**   | **同步 (Synchronous)**                                                    | **异步 (Asynchronous)**                                                                      |
| **阻塞特性**     | **阻塞 (Blocking)**。在执行数据库读写操作时，会暂停当前线程，直到操作完成。                           | **非阻塞 (Non-blocking)**。在等待数据库读写操作时，可以将控制权交给事件循环，允许其他任务并行执行。                                |
| **适用场景**     | 适用于传统的同步代码，或当你使用 **`graph.invoke()`** 或 **`graph.stream()`** 等同步方法执行图时。 | 适用于现代的异步代码（如 `async/await`），或当你使用 **`graph.ainvoke()`** 或 **`graph.astream()`** 等异步方法执行图时。 |
| **底层库**      | 依赖 Python 标准库的 **`sqlite3`** 模块。                                        | 依赖 **`aiosqlite`** 库，该库为 `sqlite3` 提供了异步接口。                                                |
| **Agent 匹配** | 必须与同步运行的 Agent 实例（例如您之前的 `abot`）配合使用。                                   | 必须与异步运行的 Agent 实例（例如您之前的 `abot_async`）配合使用。                                                |



In [64]:
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
import aiosqlite

# 需要搭配with使用参考上面的SqliteSaver部分的说明
# checkpointer = AsyncSqliteSaver.from_conn_string(":memory:")
aio_conn = await aiosqlite.connect(":memory:")
aio_checkpointer = AsyncSqliteSaver(aio_conn)

abot = Agent(model, [tool], system=prompt, checkpointer=aio_checkpointer)

In [ ]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}

# astream_events：监听所有底层事件（开始、流式、结束），适合调试和详细监控
async for event in abot.graph.astream_events(
    {"messages": messages}, 
    thread,
):
    # 监听模型开始事件
    if event["event"] == "on_chat_model_start":
        print(f"{event['data']['input']}\nbegin\n")

    # 监听模型流式输出事件（每个 token）
    elif event["event"] == "on_chat_model_stream":
        print(event['data']['chunk'].text, end="|")

    # 监听模型结束事件
    elif event["event"] == "on_chat_model_end":
        print(f"\{event['data']['output'].text}\nend\n")

    else:
        pass

# 使用 astream 的 messages 模式流式输出
async for token, metadata in abot.graph.astream(
    {"messages": messages}, 
    thread, 
    stream_mode="messages",  # 专门用于流式输出 LLM tokens
):
    # 打印每个 token
    if token.content:
        print(token.content, end="|", flush=True)

Calling: {'name': 'tavily_search', 'args': {'query': 'What is the weather in SF?'}, 'id': 'call_62e85961e9e54ca2942d2f', 'type': 'tool_call'}
Back to the model!
{'query': 'What is the weather in SF?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Weather in San Francisco', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1763102808, 'localtime': '2025-11-13 22:46'}, 'current': {'last_updated_epoch': 1763102700, 'last_updated': '2025-11-13 22:45', 'temp_c': 15.6, 'temp_f': 60.1, 'is_day': 0, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/night/116.png', 'code': 1003}, 'wind_mph': 6.3, 'wind_kph': 10.1, 'wind_degree': 168, 'wind_dir': 'SSE', 'pressure_mb': 1013.0, 'pressure_in': 29.9, 'precip_mm': 0.25, 'precip_in': 0.01, 'humidity':


stream_mode 的不同值详解：

官网：https://docs.langchain.com/oss/python/langgraph/streaming

1. values
+ 返回内容：每个超级步骤（super-step）后的完整状态
+ 使用场景：需要查看整个图状态的完整快照
+ 示例：获取所有消息、所有状态字段
2. updates
+ 返回内容：每个节点执行后的状态更新（增量变化）
+ 使用场景：追踪每个节点对状态的具体修改
+ 示例：{"agent": {"messages": [新消息]}}  ： {"agent": {"messages": [新消息]}}
3. messages
+ 返回内容：LLM 生成的 token + 元数据（二元组）
+ 使用场景：聊天应用中逐 token 显示 LLM 输出
+ 示例：(token, {"langgraph_node": "agent"})  ： (token, {"langgraph_node": "agent"})
4. custom
+ 返回内容：节点内通过 get_stream_writer() 发送的自定义数据
+ 使用场景：工具执行进度、自定义事件通知
+ 示例："正在查询数据库..."
5. debug
+ 返回内容：尽可能多的调试信息
+ 使用场景：开发调试、问题排查